# FAOSTAT Olive Production - Feature Engineering

**Module Project 1: Data Storytelling**  
**Alberto Gomez Soteres & Burak Donbekci**

This notebook creates new features from the cleaned FAOSTAT data to better understand how olive production has changed in Spain and Türkiye over time.

The engineered features focus on growth, long-term trends, and changes in production, harvested area, and yield.

## 1. Imports

First, we import the library needed to work with the cleaned datasets.

In [1]:
from pathlib import Path

import pandas as pd


# Locate the project root
project_root = Path.cwd()

if not (project_root / "data").exists():
    project_root = project_root.parent

if not (project_root / "data").exists():
    raise FileNotFoundError("Project root could not be found.")

print("Project root:", project_root)

Project root: /Users/alberto/Library/CloudStorage/OneDrive-DukeUniversity/Duke/Academics/AIPI 510 Sourcing/Project 1 repo/Big-Olive


## 2. Load the Clean Data

Next, we load the cleaned FAOSTAT dataset for Spain and Türkiye from the project's processed data directory.

In [2]:
# Load the cleaned FAOSTAT dataset
file_path = (
    project_root
    / "data"
    / "processed"
    / "faostat_clean.csv"
)

df = pd.read_csv(
    file_path,
    dtype={"m49_code": "string"},
)

df.head()

,faostat_country_code,m49_code,country,year,production_tonnes,production_flag,area_harvested_ha,area_harvested_flag,yield_kg_ha,yield_flag
0,203,724,Spain,1961,1863400.0,A,NaN,M,NaN,NaN
1,203,724,Spain,1962,1641000.0,A,NaN,M,NaN,NaN
2,203,724,Spain,1963,3124300.0,A,NaN,M,NaN,NaN
3,203,724,Spain,1964,572700.0,A,NaN,M,NaN,NaN
4,203,724,Spain,1965,1656100.0,A,NaN,M,NaN,NaN


In [3]:
# Check the dataset shape
print("Dataset shape:", df.shape)

Dataset shape: (128, 10)


## 3. Production Growth

First, we calculate the year-over-year percentage change in olive production for each country. This helps us identify periods of strong growth or decline that may be difficult to see from production values alone.

In [4]:
# Calculate annual production growth for each country
df["production_growth_pct"] = (
    df.groupby("country")["production_tonnes"]
    .pct_change(fill_method=None)
    * 100
)

# Preview the new feature
df[
    ["country", "year", "production_tonnes", "production_growth_pct"]
].head(10)

,country,year,production_tonnes,production_growth_pct
0,Spain,1961,1863400.0,NaN
1,Spain,1962,1641000.0,-11.935172
2,Spain,1963,3124300.0,90.390006
3,Spain,1964,572700.0,-81.669494
4,Spain,1965,1656100.0,189.174088
5,Spain,1966,2101500.0,26.894511
6,Spain,1967,1372000.0,-34.713300
7,Spain,1968,2274300.0,65.765306
8,Spain,1969,1738300.0,-23.567691
9,Spain,1970,2100000.0,20.807686


In [5]:
# Look at the most recent production changes
df[
    ["country", "year", "production_tonnes", "production_growth_pct"]
].groupby("country").tail(5)

,country,year,production_tonnes,production_growth_pct
59,Spain,2020,8137810.0,36.424155
60,Spain,2021,8256550.0,1.459115
61,Spain,2022,3940070.0,-52.279463
62,Spain,2023,5101010.0,29.464959
63,Spain,2024,8310250.0,62.913815
123,Türkiye,2020,1316626.0,-13.663869
124,Türkiye,2021,1738680.0,32.055724
125,Türkiye,2022,2976000.0,71.164332
126,Türkiye,2023,1520000.0,-48.924731
127,Türkiye,2024,3750000.0,146.710526


The recent values show strong year-to-year variability in olive production, especially in Türkiye. This feature will help us highlight periods of unusually high growth or decline in the final analysis.

## 4. Five-Year Production Trend

Next, we calculate a five-year rolling average of olive production for each country. This smooths short-term fluctuations and makes the longer-term production trend easier to see.

In [6]:
# Calculate the five-year rolling average for each country
df["production_5yr_avg"] = (
    df.groupby("country")["production_tonnes"]
    .transform(lambda x: x.rolling(window=5, min_periods=5).mean())
)

# Preview the new feature
df[
    ["country", "year", "production_tonnes", "production_5yr_avg"]
].head(10)

,country,year,production_tonnes,production_5yr_avg
0,Spain,1961,1863400.0,NaN
1,Spain,1962,1641000.0,NaN
2,Spain,1963,3124300.0,NaN
3,Spain,1964,572700.0,NaN
4,Spain,1965,1656100.0,1771500.0
5,Spain,1966,2101500.0,1819120.0
6,Spain,1967,1372000.0,1765320.0
7,Spain,1968,2274300.0,1595320.0
8,Spain,1969,1738300.0,1828440.0
9,Spain,1970,2100000.0,1917220.0


In [7]:
# Check the most recent five-year production trends
df[
    ["country", "year", "production_tonnes", "production_5yr_avg"]
].groupby("country").tail(5)

,country,year,production_tonnes,production_5yr_avg
59,Spain,2020,8137810.0,7510901.8
60,Spain,2021,8256550.0,7745701.8
61,Spain,2022,3940070.0,7223816.0
62,Spain,2023,5101010.0,6280104.0
63,Spain,2024,8310250.0,6749138.0
123,Türkiye,2020,1316626.0,1634418.6
124,Türkiye,2021,1738680.0,1636154.6
125,Türkiye,2022,2976000.0,1811354.6
126,Türkiye,2023,1520000.0,1815261.2
127,Türkiye,2024,3750000.0,2260261.2


The rolling average reduces the effect of individual high or low production years and provides a clearer view of the longer-term trend.

## 5. Indexed Production

To compare relative production growth between Spain and Türkiye, we index production to a common baseline.

Because annual olive production can vary strongly from year to year, we use the average production from 1961 to 1965 as the baseline instead of relying on a single year.

In [8]:
# Calculate the average production from 1961 to 1965 for each country
base_production = (
    df[df["year"].between(1961, 1965)]
    .groupby("country")["production_tonnes"]
    .mean()
)

# Calculate indexed production with the 1961–1965 average = 100
df["production_index"] = (
    df["production_tonnes"]
    / df["country"].map(base_production)
    * 100
)

df[
    ["country", "year", "production_tonnes", "production_index"]
].head(10)

,country,year,production_tonnes,production_index
0,Spain,1961,1863400.0,105.187694
1,Spain,1962,1641000.0,92.633362
2,Spain,1963,3124300.0,176.364663
3,Spain,1964,572700.0,32.328535
4,Spain,1965,1656100.0,93.485747
5,Spain,1966,2101500.0,118.628281
6,Spain,1967,1372000.0,77.448490
7,Spain,1968,2274300.0,128.382727
8,Spain,1969,1738300.0,98.125882
9,Spain,1970,2100000.0,118.543607


In [9]:
# Check the average index during the baseline period
df[df["year"].between(1961, 1965)].groupby("country")[
    "production_index"
].mean()

country
Spain      100.0
Türkiye    100.0
Name: production_index, dtype: float64

In [10]:
# Check the indexed production values in 2024
df[df["year"] == 2024][
    ["country", "production_tonnes", "production_index"]
]

,country,production_tonnes,production_index
63,Spain,8310250.0,469.108100
127,Türkiye,3750000.0,697.803464


The production index allows us to compare how much each country's production has changed relative to its own starting level, instead of comparing only absolute production volumes.

## 6. Harvested Area and Yield Growth

Since Spain has missing harvested area and yield values before 1980, we use 1980–2024 for these comparisons.

We calculate the year-over-year percentage change in harvested area and yield for each country.

In [11]:
# Keep the common period with complete area and yield data
area_yield_df = df[df["year"] >= 1980].copy()

# Calculate annual growth in harvested area
area_yield_df["area_growth_pct"] = (
    area_yield_df.groupby("country")["area_harvested_ha"]
    .pct_change(fill_method=None)
    * 100
)

# Calculate annual growth in yield
area_yield_df["yield_growth_pct"] = (
    area_yield_df.groupby("country")["yield_kg_ha"]
    .pct_change(fill_method=None)
    * 100
)

area_yield_df[
    [
        "country",
        "year",
        "area_harvested_ha",
        "area_growth_pct",
        "yield_kg_ha",
        "yield_growth_pct",
    ]
].head(10)

,country,year,area_harvested_ha,area_growth_pct,yield_kg_ha,yield_growth_pct
19,Spain,1980,1156500.0,NaN,1949.8,NaN
20,Spain,1981,2045000.0,76.826632,743.7,-61.857626
21,Spain,1982,2045800.0,0.039120,1631.6,119.389539
22,Spain,1983,2050400.0,0.224851,647.9,-60.290512
23,Spain,1984,2039100.0,-0.551112,1728.9,166.846736
24,Spain,1985,2051000.0,0.583591,970.0,-43.894962
25,Spain,1986,2063000.0,0.585080,1239.4,27.773196
26,Spain,1987,2057000.0,-0.290839,1885.8,52.154268
27,Spain,1988,2035000.0,-1.069519,1092.7,-42.056422
28,Spain,1989,2054000.0,0.933661,1434.1,31.243708


In [12]:
# Check the most recent changes
area_yield_df[
    [
        "country",
        "year",
        "area_growth_pct",
        "yield_growth_pct",
    ]
].groupby("country").tail(5)

,country,year,area_growth_pct,yield_growth_pct
59,Spain,2020,0.838618,35.287447
60,Spain,2021,-0.016389,1.476657
61,Spain,2022,0.457060,-52.497299
62,Spain,2023,0.600695,28.693733
63,Spain,2024,-0.204820,63.250351
123,Türkiye,2020,0.898568,-14.435605
124,Türkiye,2021,0.235718,31.747743
125,Türkiye,2022,1.344853,68.891275
126,Türkiye,2023,0.214731,-49.032551
127,Türkiye,2024,1.135914,143.934173


Using a common period avoids comparing Spain and Türkiye with different data coverage. These features help separate changes related to harvested area from changes in output per hectare.

## 7. Production Growth Decomposition

Because FAOSTAT yield is derived from production and harvested area, annual production growth can be decomposed into changes in area, changes in yield, and their interaction.

This is an accounting decomposition, not a causal model.

In [13]:
# Calculate the interaction between area and yield growth
area_yield_df["growth_interaction_pct"] = (
    area_yield_df["area_growth_pct"]
    * area_yield_df["yield_growth_pct"]
    / 100
)

# Reconstruct production growth from its components
area_yield_df["reconstructed_production_growth_pct"] = (
    area_yield_df["area_growth_pct"]
    + area_yield_df["yield_growth_pct"]
    + area_yield_df["growth_interaction_pct"]
)

# Compare the reconstructed growth with the actual production growth
area_yield_df["growth_difference_pct"] = (
    area_yield_df["production_growth_pct"]
    - area_yield_df["reconstructed_production_growth_pct"]
)

area_yield_df[
    [
        "country",
        "year",
        "production_growth_pct",
        "area_growth_pct",
        "yield_growth_pct",
        "growth_interaction_pct",
        "reconstructed_production_growth_pct",
    ]
].head(10)

,country,year,production_growth_pct,area_growth_pct,yield_growth_pct,growth_interaction_pct,reconstructed_production_growth_pct
19,Spain,1980,-1.960784,NaN,NaN,NaN,NaN
20,Spain,1981,-32.558758,76.826632,-61.857626,-47.523131,-32.554125
21,Spain,1982,119.483167,0.039120,119.389539,0.046705,119.475364
22,Spain,1983,-60.202523,0.224851,-60.290512,-0.135564,-60.201225
23,Spain,1984,165.386932,-0.551112,166.846736,-0.919512,165.376111
24,Spain,1985,-43.566687,0.583591,-43.894962,-0.256167,-43.567538
25,Spain,1986,28.514702,0.585080,27.773196,0.162496,28.520772
26,Spain,1987,51.713079,-0.290839,52.154268,-0.151685,51.711745
27,Spain,1988,-42.673369,-1.069519,-42.056422,0.449801,-42.676139
28,Spain,1989,32.468409,0.933661,31.243708,0.291710,32.469079


In [14]:
# Check the maximum difference between actual and reconstructed growth
max_difference = area_yield_df["growth_difference_pct"].abs().max()

print(f"Maximum absolute difference: {max_difference:.4f} percentage points")

Maximum absolute difference: 0.0117 percentage points


The reconstructed values closely match the observed production growth. This confirms that changes in production can be described through changes in harvested area and yield, with a small interaction effect.

## 8. Production Share Among Major Producers

To add context to the Spain–Türkiye comparison, we calculate each country's production share within a consistent group of major olive producers.

We first identify the ten countries with the highest cumulative olive production from 1961 to 2024. To keep the denominator consistent across the full period, we then retain only countries with complete annual production coverage.

This measure represents production share within this comparison group, not world production share.

In [15]:
# Load the cleaned dataset containing all available countries
all_countries_path = (
    project_root
    / "data"
    / "processed"
    / "faostat_all_countries.csv"
)

all_countries_df = pd.read_csv(
    all_countries_path,
    dtype={"m49_code": "string"},
)

all_countries_df.head()

,faostat_country_code,m49_code,country,year,production_tonnes,production_flag,area_harvested_ha,area_harvested_flag,yield_kg_ha,yield_flag
0,2,004,Afghanistan,1961,1000.0,E,600.0,E,1666.7,E
1,2,004,Afghanistan,1962,1100.0,E,600.0,E,1833.3,E
2,2,004,Afghanistan,1963,1000.0,E,600.0,E,1666.7,E
3,2,004,Afghanistan,1964,1100.0,E,600.0,E,1833.3,E
4,2,004,Afghanistan,1965,1000.0,E,600.0,E,1666.7,E


In [16]:
# Identify the ten countries with the highest cumulative production
cumulative_production = (
    all_countries_df.groupby("country", as_index=False)["production_tonnes"]
    .sum(min_count=1)
    .sort_values("production_tonnes", ascending=False)
)

top_10_producers = cumulative_production.head(10)["country"].tolist()

top_10_producers

['Spain',
 'Italy',
 'Greece',
 'Türkiye',
 'Tunisia',
 'Morocco',
 'Syrian Arab Republic',
 'Portugal',
 'Algeria',
 'Egypt']

In [17]:
# Check annual production coverage within the top ten producers
total_years = all_countries_df["year"].nunique()

top_10_df = all_countries_df[
    all_countries_df["country"].isin(top_10_producers)
].copy()

coverage = (
    top_10_df.groupby("country")
    .agg(
        years_available=("year", "nunique"),
        production_years=("production_tonnes", "count"),
    )
    .reset_index()
)

coverage["complete_coverage"] = (
    coverage["production_years"] == total_years
)

coverage.sort_values("production_years", ascending=False)

,country,years_available,production_years,complete_coverage
0,Algeria,64,64,True
1,Egypt,64,64,True
3,Italy,64,64,True
4,Morocco,64,64,True
5,Portugal,64,64,True
6,Spain,64,64,True
7,Syrian Arab Republic,64,64,True
8,Tunisia,64,64,True
9,Türkiye,64,64,True
2,Greece,64,63,False


In [18]:
# Keep only major producers with complete production coverage
complete_producers = coverage.loc[
    coverage["complete_coverage"],
    "country"
].tolist()

print("Countries in comparison group:", len(complete_producers))
print(complete_producers)

Countries in comparison group: 9
['Algeria', 'Egypt', 'Italy', 'Morocco', 'Portugal', 'Spain', 'Syrian Arab Republic', 'Tunisia', 'Türkiye']


In [19]:
# Calculate total annual production for the consistent comparison group
major_df = top_10_df[
    top_10_df["country"].isin(complete_producers)
].copy()

major_totals = (
    major_df.groupby("year")["production_tonnes"]
    .sum()
)

# Add the comparison-group total to Spain and Türkiye
df["major_production_total"] = df["year"].map(major_totals)

# Calculate production share within the comparison group
df["major_producer_share_pct"] = (
    df["production_tonnes"]
    / df["major_production_total"]
    * 100
)

df[
    [
        "country",
        "year",
        "production_tonnes",
        "major_producer_share_pct",
    ]
].head(10)

,country,year,production_tonnes,major_producer_share_pct
0,Spain,1961,1863400.0,29.814934
1,Spain,1962,1641000.0,35.258476
2,Spain,1963,3124300.0,37.795802
3,Spain,1964,572700.0,13.127294
4,Spain,1965,1656100.0,29.436989
5,Spain,1966,2101500.0,37.361587
6,Spain,1967,1372000.0,23.498897
7,Spain,1968,2274300.0,35.159364
8,Spain,1969,1738300.0,30.600382
9,Spain,1970,2100000.0,33.870558


In [20]:
# Check production share at selected points in time
df[df["year"].isin([1961, 1990, 2020, 2023, 2024])][
    ["country", "year", "major_producer_share_pct"]
]

,country,year,major_producer_share_pct
0,Spain,1961,29.814934
29,Spain,1990,45.038729
59,Spain,2020,43.668756
62,Spain,2023,33.801117
63,Spain,2024,39.942874
64,Türkiye,1961,11.029382
93,Türkiye,1990,14.704123
123,Türkiye,2020,7.065220
126,Türkiye,2023,10.072064
127,Türkiye,2024,18.024220


The comparison group contains nine of the ten largest cumulative producers. Greece is excluded because its production series is incomplete in 2024, allowing the same denominator to be used consistently from 1961 to 2024.

This feature should therefore be interpreted as share within the selected comparison group, not as world production share.

## 9. Production Direction Alternation

Olive production can alternate between increases and decreases from one year to the next.

To capture this pattern, we first classify each year as an increase or decrease in production. We then create a feature indicating whether the direction changed compared with the previous year.

This feature is descriptive. Statistical testing of the alternation pattern will be performed later in the analysis stage.

In [21]:
# Classify the direction of annual production change
df["production_direction"] = pd.NA

df.loc[
    df["production_growth_pct"] > 0,
    "production_direction"
] = "increase"

df.loc[
    df["production_growth_pct"] < 0,
    "production_direction"
] = "decrease"

df.loc[
    df["production_growth_pct"] == 0,
    "production_direction"
] = "no_change"

In [22]:
# Compare each year's direction with the previous year
previous_direction = (
    df.groupby("country")["production_direction"]
    .shift(1)
)

# Create a nullable Boolean feature for direction changes
df["alternating_direction"] = pd.Series(
    pd.NA,
    index=df.index,
    dtype="boolean"
)

valid_direction = (
    df["production_direction"].isin(["increase", "decrease"])
    & previous_direction.isin(["increase", "decrease"])
)

df.loc[
    valid_direction,
    "alternating_direction"
] = (
    df.loc[valid_direction, "production_direction"]
    != previous_direction[valid_direction]
)

In [23]:
# Preview the alternation feature
df[
    [
        "country",
        "year",
        "production_growth_pct",
        "production_direction",
        "alternating_direction",
    ]
].groupby("country").head(8)

,country,year,production_growth_pct,production_direction,alternating_direction
0,Spain,1961,NaN,<NA>,<NA>
1,Spain,1962,-11.935172,decrease,<NA>
2,Spain,1963,90.390006,increase,True
3,Spain,1964,-81.669494,decrease,True
4,Spain,1965,189.174088,increase,True
5,Spain,1966,26.894511,increase,False
6,Spain,1967,-34.713300,decrease,True
7,Spain,1968,65.765306,increase,True
64,Türkiye,1961,NaN,<NA>,<NA>
65,Türkiye,1962,-57.902235,decrease,<NA>


In [24]:
# Calculate the percentage of valid years that alternate direction
alternation_summary = (
    df.dropna(subset=["alternating_direction"])
    .groupby("country")["alternating_direction"]
    .mean()
    .mul(100)
)

alternation_summary

country
Spain      69.354839
Türkiye    91.935484
Name: alternating_direction, dtype: Float64

The `alternating_direction` feature identifies years in which production switches from an increase to a decrease, or from a decrease to an increase.

The percentage above is only a descriptive summary. We will evaluate whether the observed alternation is stronger than expected by chance during the analysis and robustness stage.

## 10. Combine and Validate Engineered Features

Before saving the final dataset, we add the area and yield features back to the main dataset and check that the engineered features were created correctly.

In [25]:
# Add the area and yield features back to the main dataset
feature_columns = [
    "area_growth_pct",
    "yield_growth_pct",
    "growth_interaction_pct",
    "reconstructed_production_growth_pct",
    "growth_difference_pct",
]

df.loc[
    area_yield_df.index,
    feature_columns
] = area_yield_df[feature_columns]

df.head()

,faostat_country_code,m49_code,country,year,production_tonnes,production_flag,area_harvested_ha,area_harvested_flag,yield_kg_ha,yield_flag,...,production_index,major_production_total,major_producer_share_pct,production_direction,alternating_direction,area_growth_pct,yield_growth_pct,growth_interaction_pct,reconstructed_production_growth_pct,growth_difference_pct
0,203,724,Spain,1961,1863400.0,A,NaN,M,NaN,NaN,...,105.187694,6249888.0,29.814934,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN
1,203,724,Spain,1962,1641000.0,A,NaN,M,NaN,NaN,...,92.633362,4654200.0,35.258476,decrease,<NA>,NaN,NaN,NaN,NaN,NaN
2,203,724,Spain,1963,3124300.0,A,NaN,M,NaN,NaN,...,176.364663,8266262.0,37.795802,increase,True,NaN,NaN,NaN,NaN,NaN
3,203,724,Spain,1964,572700.0,A,NaN,M,NaN,NaN,...,32.328535,4362666.0,13.127294,decrease,True,NaN,NaN,NaN,NaN,NaN
4,203,724,Spain,1965,1656100.0,A,NaN,M,NaN,NaN,...,93.485747,5625915.0,29.436989,increase,True,NaN,NaN,NaN,NaN,NaN


In [26]:
# Check the final dataset structure
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (128, 22)

Columns:
['faostat_country_code', 'm49_code', 'country', 'year', 'production_tonnes', 'production_flag', 'area_harvested_ha', 'area_harvested_flag', 'yield_kg_ha', 'yield_flag', 'production_growth_pct', 'production_5yr_avg', 'production_index', 'major_production_total', 'major_producer_share_pct', 'production_direction', 'alternating_direction', 'area_growth_pct', 'yield_growth_pct', 'growth_interaction_pct', 'reconstructed_production_growth_pct', 'growth_difference_pct']


In [27]:
# Check missing values in the engineered features
engineered_features = [
    "production_growth_pct",
    "production_5yr_avg",
    "production_index",
    "area_growth_pct",
    "yield_growth_pct",
    "growth_interaction_pct",
    "reconstructed_production_growth_pct",
    "growth_difference_pct",
    "major_producer_share_pct",
    "production_direction",
    "alternating_direction",
]

df[engineered_features].isna().sum()

production_growth_pct                   2
production_5yr_avg                      8
production_index                        0
area_growth_pct                        40
yield_growth_pct                       40
growth_interaction_pct                 40
reconstructed_production_growth_pct    40
growth_difference_pct                  40
major_producer_share_pct                0
production_direction                    2
alternating_direction                   4
dtype: int64

## 11. Save Feature Data

Finally, we save the dataset with the engineered features so it can be used in the visualization and analysis stage of the project.

In [28]:
# Save the final feature dataset
output_path = (
    project_root
    / "data"
    / "processed"
    / "faostat_features.csv"
)

df.to_csv(output_path, index=False)

print("Feature dataset saved successfully.")
print("Final shape:", df.shape)

Feature dataset saved successfully.
Final shape: (128, 22)
